# neural_net_walk_forward_validation_tuning

MLP validation tuning using the full-history session-aligned dataset.

The notebook tests compact neural-network hyperparameter settings and small feature-set variants, including alternative z-score windows, percentile transforms, clipped z-scores, missingness flags, and one-day lagged signal variants. Selection is based only on walk-forward validation, then the selected configurations are evaluated on the untouched test split.


In [1]:
from __future__ import annotations

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 260)


In [2]:
from __future__ import annotations

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "datasets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from notebook_utils.experiment_config import build_default_config
from notebook_utils.feature_set_grid_builder import FeatureFrameBuilder, FeatureSetGridBuilder
from notebook_utils.metrics import ClassificationMetrics
from notebook_utils.model_report_builder import ModelReportBuilder
from notebook_utils.split_utils import make_split_dates, make_walk_forward_fold_specs, subset_by_dates

CONFIG = build_default_config(PROJECT_ROOT)

MLP_PARAM_GRID = [
    {
        "param_set": "mlp_8_alpha1e-3_lr1e-3_b128",
        "hidden_layer_sizes": (8,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-3,
        "learning_rate_init": 1e-3,
        "batch_size": 128,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
    {
        "param_set": "mlp_16_alpha1e-3_lr1e-3_b128",
        "hidden_layer_sizes": (16,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-3,
        "learning_rate_init": 1e-3,
        "batch_size": 128,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
    {
        "param_set": "mlp_16_alpha1e-2_lr1e-3_b128",
        "hidden_layer_sizes": (16,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-2,
        "learning_rate_init": 1e-3,
        "batch_size": 128,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
    {
        "param_set": "mlp_16_8_alpha1e-3_lr1e-3_b128",
        "hidden_layer_sizes": (16, 8),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-3,
        "learning_rate_init": 1e-3,
        "batch_size": 128,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
    {
        "param_set": "mlp_32_16_alpha1e-3_lr5e-4_b128",
        "hidden_layer_sizes": (32, 16),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-3,
        "learning_rate_init": 5e-4,
        "batch_size": 128,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
    {
        "param_set": "mlp_16_alpha1e-4_lr1e-3_b256",
        "hidden_layer_sizes": (16,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-4,
        "learning_rate_init": 1e-3,
        "batch_size": 256,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
    {
        "param_set": "mlp_32_alpha1e-3_lr1e-3_b256",
        "hidden_layer_sizes": (32,),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-3,
        "learning_rate_init": 1e-3,
        "batch_size": 256,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
    {
        "param_set": "mlp_32_16_alpha1e-2_lr5e-4_b256",
        "hidden_layer_sizes": (32, 16),
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-2,
        "learning_rate_init": 5e-4,
        "batch_size": 256,
        "max_iter": 500,
        "early_stopping": False,
        "n_iter_no_change": 25,
        "tol": 1e-4,
    },
]

pd.DataFrame(MLP_PARAM_GRID)

feature_grid = FeatureSetGridBuilder.build(
    max_features_per_model=CONFIG["max_features_per_model"],
)

PRICE_FEATURES = feature_grid.price_features
VOLUME_FEATURE_OPTIONS = feature_grid.volume_feature_options
GDELT_FEATURE_OPTIONS = feature_grid.gdelt_feature_options
GDELT_SENTIMENT_FEATURE_OPTIONS = feature_grid.gdelt_sentiment_feature_options
GDELT_ATTENTION_FEATURE_OPTIONS = feature_grid.gdelt_attention_feature_options
REDDIT_FEATURE_OPTIONS = feature_grid.reddit_feature_options
REDDIT_ATTENTION_FEATURE_OPTIONS = feature_grid.reddit_attention_feature_options
GOOGLE_TRENDS_FEATURE_OPTIONS = feature_grid.google_trends_feature_options
GOOGLE_SCORE_ATTENTION_FEATURE_OPTIONS = feature_grid.google_score_attention_feature_options
DERIVED_FEATURE_COLUMNS = feature_grid.derived_feature_columns
BASE_VOLUME_OPTION = feature_grid.base_volume_option
BASELINE_FEATURE_SET = feature_grid.baseline_feature_set
FEATURE_SET_SPECS = feature_grid.feature_set_specs
FEATURE_SETS = feature_grid.feature_sets
FEATURE_SET_METADATA = feature_grid.feature_set_metadata
SKIPPED_FEATURE_SETS = feature_grid.skipped_feature_sets
FEATURE_SETS_TO_TEST = feature_grid.feature_sets_to_test
pd.DataFrame(MLP_PARAM_GRID)


,param_set,hidden_layer_sizes,activation,solver,alpha,learning_rate_init,batch_size,max_iter,early_stopping,n_iter_no_change,tol
0,mlp_8_alpha1e-3_lr1e-3_b128,"(8,)",relu,adam,0.0010,0.0010,128,500,False,25,0.0001
1,mlp_16_alpha1e-3_lr1e-3_b128,"(16,)",relu,adam,0.0010,0.0010,128,500,False,25,0.0001
2,mlp_16_alpha1e-2_lr1e-3_b128,"(16,)",relu,adam,0.0100,0.0010,128,500,False,25,0.0001
3,mlp_16_8_alpha1e-3_lr1e-3_b128,"(16, 8)",relu,adam,0.0010,0.0010,128,500,False,25,0.0001
4,mlp_32_16_alpha1e-3_lr5e-4_b128,"(32, 16)",relu,adam,0.0010,0.0005,128,500,False,25,0.0001
5,mlp_16_alpha1e-4_lr1e-3_b256,"(16,)",relu,adam,0.0001,0.0010,256,500,False,25,0.0001
6,mlp_32_alpha1e-3_lr1e-3_b256,"(32,)",relu,adam,0.0010,0.0010,256,500,False,25,0.0001
7,mlp_32_16_alpha1e-2_lr5e-4_b256,"(32, 16)",relu,adam,0.0100,0.0005,256,500,False,25,0.0001


In [3]:
from __future__ import annotations



def build_mlp_pipeline_from_params(params: dict) -> Pipeline:
    mlp_params = dict(params)
    mlp_params.pop("param_set", None)
    mlp_params["hidden_layer_sizes"] = tuple(mlp_params["hidden_layer_sizes"])
    mlp_params.setdefault("random_state", CONFIG["random_state"])
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(**mlp_params)),
        ]
    )


In [4]:
raw_df = pd.read_csv(CONFIG["dataset_path"], parse_dates=["date"])
raw_df = raw_df[~raw_df["ticker"].isin(CONFIG["excluded_tickers"])].copy()
raw_df = raw_df.sort_values(["ticker", "date"]).reset_index(drop=True)

feature_df = FeatureFrameBuilder.build_feature_frame(raw_df, neutral_band=CONFIG["neutral_band"])
train_dates, validation_dates, test_dates = make_split_dates(
    feature_df,
    test_size=CONFIG["test_size"],
    validation_fraction_within_pretest=CONFIG["validation_fraction_within_pretest"],
    min_validation_dates=CONFIG["min_validation_dates"],
    gap_days=CONFIG["gap_days"],
)
walk_forward_fold_specs = make_walk_forward_fold_specs(
    train_dates,
    n_folds=CONFIG["walk_forward_folds"],
    validation_size=CONFIG["walk_forward_validation_dates"],
    min_train_dates=CONFIG["walk_forward_min_train_dates"],
    gap_days=CONFIG["gap_days"],
)

split_date_map = {
    "train": train_dates,
    "validation": validation_dates,
    "test": test_dates,
}
feature_df["split"] = "gap"
for split_name, split_dates in split_date_map.items():
    feature_df.loc[feature_df["date"].isin(split_dates), "split"] = split_name

modeled_df = feature_df[feature_df["target"].isin([0.0, 1.0])].copy()
modeled_df["target"] = modeled_df["target"].astype(int)

train_df = subset_by_dates(modeled_df, train_dates)
validation_df = subset_by_dates(modeled_df, validation_dates)
test_df = subset_by_dates(modeled_df, test_dates)
train_validation_df = subset_by_dates(modeled_df, list(train_dates) + list(validation_dates))

split_summary_rows = []
for split_name in ["train", "validation", "test"]:
    all_split_df = feature_df[feature_df["split"].eq(split_name)]
    modeled_split_df = modeled_df[modeled_df["split"].eq(split_name)]
    split_summary_rows.append(
        {
            "split": split_name,
            "session_rows": len(all_split_df),
            "modeled_rows": len(modeled_split_df),
            "session_dates": all_split_df["date"].nunique(),
            "modeled_dates": modeled_split_df["date"].nunique(),
            "date_min": all_split_df["date"].min(),
            "date_max": all_split_df["date"].max(),
            "target_positive_rate": modeled_split_df["target"].mean(),
        }
    )

split_summary_df = pd.DataFrame(split_summary_rows)
walk_forward_fold_summary_df = pd.DataFrame(
    [
        {
            "fold": spec["fold"],
            "train_n_dates": spec["train_n_dates"],
            "train_date_min": spec["train_date_min"],
            "train_date_max": spec["train_date_max"],
            "validation_n_dates": spec["validation_n_dates"],
            "validation_date_min": spec["validation_date_min"],
            "validation_date_max": spec["validation_date_max"],
        }
        for spec in walk_forward_fold_specs
    ]
)

split_summary_df


,split,session_rows,modeled_rows,session_dates,modeled_dates,date_min,date_max,target_positive_rate
0,train,5632,4456,704,704,2021-01-04,2023-10-19,0.511670
1,validation,1880,1424,235,235,2023-10-23,2024-09-27,0.568118
2,test,2512,1886,314,313,2024-10-01,2025-12-31,0.526511


In [5]:
walk_forward_fold_summary_df


,fold,train_n_dates,train_date_min,train_date_max,validation_n_dates,validation_date_min,validation_date_max
0,1,383,2021-01-04,2022-07-12,80,2022-07-14,2022-11-03
1,2,463,2021-01-04,2022-11-02,80,2022-11-04,2023-03-02
2,3,543,2021-01-04,2023-03-01,80,2023-03-03,2023-06-27
3,4,623,2021-01-04,2023-06-26,80,2023-06-28,2023-10-19


In [6]:
neutral_summary_by_ticker_df = (
    feature_df.groupby("ticker")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reset_index()
)
neutral_summary_by_ticker_df["neutral_rate_among_available"] = (
    neutral_summary_by_ticker_df["neutral"] / neutral_summary_by_ticker_df["target_available"]
)
neutral_summary_by_ticker_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_ticker_df["neutral_rate_among_available"]

neutral_summary_by_split_df = (
    feature_df[feature_df["split"].isin(["train", "validation", "test"])]
    .groupby("split")
    .agg(
        rows=("target_available", "size"),
        target_available=("target_available", "sum"),
        neutral=("is_neutral", "sum"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
neutral_summary_by_split_df["neutral_rate_among_available"] = (
    neutral_summary_by_split_df["neutral"] / neutral_summary_by_split_df["target_available"]
)
neutral_summary_by_split_df["modeled_rate_among_available"] = 1.0 - neutral_summary_by_split_df["neutral_rate_among_available"]

print("Neutral coverage by split")
print(neutral_summary_by_split_df.to_string(index=False))
print("\nNeutral coverage by ticker")
neutral_summary_by_ticker_df


Neutral coverage by split
     split  rows  target_available  neutral  neutral_rate_among_available  modeled_rate_among_available
     train  5632              5632     1176                      0.208807                      0.791193
validation  1880              1880      456                      0.242553                      0.757447
      test  2512              2504      618                      0.246805                      0.753195

Neutral coverage by ticker


,ticker,rows,target_available,neutral,neutral_rate_among_available,modeled_rate_among_available
0,AAPL,1255,1254,378,0.301435,0.698565
1,AMD,1255,1254,220,0.175439,0.824561
2,AMZN,1255,1254,300,0.239234,0.760766
3,GOOGL,1255,1254,319,0.254386,0.745614
4,META,1255,1254,278,0.221691,0.778309
5,MSFT,1255,1254,400,0.318979,0.681021
6,NVDA,1255,1254,173,0.137959,0.862041
7,TSLA,1255,1254,184,0.146730,0.853270


In [7]:
missing_requested_feature_sets = [name for name in FEATURE_SETS_TO_TEST if name not in FEATURE_SETS]
if missing_requested_feature_sets:
    raise KeyError(f"Unknown feature sets: {missing_requested_feature_sets}")

missing_feature_columns = sorted(
    {
        feature
        for name in FEATURE_SETS_TO_TEST
        for feature in FEATURE_SETS[name]
        if feature not in feature_df.columns and feature not in DERIVED_FEATURE_COLUMNS
    }
)
if missing_feature_columns:
    raise KeyError(f"Missing feature columns: {missing_feature_columns}")

candidate_feature_sets_df = pd.DataFrame(
    [
        {
            "feature_set": feature_set_name,
            "feature_family": FEATURE_SET_METADATA[feature_set_name]["feature_family"],
            "n_features": len(features),
            "features": features,
        }
        for feature_set_name, features in FEATURE_SETS.items()
    ]
).sort_values(["feature_family", "n_features", "feature_set"]).reset_index(drop=True)

skipped_feature_sets_df = pd.DataFrame(SKIPPED_FEATURE_SETS)

print(f"Selection metric: {CONFIG['selection_metric']}")
print(f"Primary validation metric: {CONFIG['primary_validation_metric']}")
print(f"Walk-forward folds: {len(walk_forward_fold_specs)}")
print(f"Tune decision threshold in each validation fold: {CONFIG['tune_decision_threshold']}")
print(f"Max features per model: {CONFIG['max_features_per_model']}")
print(f"Feature sets to test: {len(FEATURE_SETS_TO_TEST)}")
print(f"Attention feature sets: {sum('attention' in FEATURE_SET_METADATA[name]['feature_family'] for name in FEATURE_SETS_TO_TEST)}")
print(f"Skipped feature sets above max feature limit: {len(SKIPPED_FEATURE_SETS)}")
print(f"MLP parameter sets: {len(MLP_PARAM_GRID)}")
print(f"Walk-forward validation fits: {len(FEATURE_SETS_TO_TEST) * len(MLP_PARAM_GRID) * len(walk_forward_fold_specs)}")

candidate_feature_sets_df


Selection metric: balanced_accuracy
Primary validation metric: balanced_accuracy
Walk-forward folds: 4
Tune decision threshold in each validation fold: True
Max features per model: 10
Feature sets to test: 80
Attention feature sets: 38
Skipped feature sets above max feature limit: 0
MLP parameter sets: 8
Walk-forward validation fits: 2560


,feature_set,feature_family,n_features,features
0,Model B - price + volume | volume log1p zscore...,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
1,Model B - price + volume | volume percentile r...,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
2,Model B - price + volume | volume zscore 10d,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
3,Model B - price + volume | volume zscore 20d,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
4,Model B - price + volume | volume zscore 20d c...,price + volume,5,"[return_1d, return_5d, return_20d, rolling_vol..."
...,...,...,...,...
75,Model F - price + volume + all alternative dat...,price + volume + all alternative data,10,"[return_1d, return_5d, return_20d, rolling_vol..."
76,Model N - price + volume + all attention | per...,price + volume + all attention,8,"[return_1d, return_5d, return_20d, rolling_vol..."
77,Model N - price + volume + all attention | zsc...,price + volume + all attention,8,"[return_1d, return_5d, return_20d, rolling_vol..."
78,Model N - price + volume + all attention | zsc...,price + volume + all attention,8,"[return_1d, return_5d, return_20d, rolling_vol..."


In [8]:
from __future__ import annotations


MLP_PARAM_COLUMNS = [
    "hidden_layer_sizes",
    "activation",
    "solver",
    "alpha",
    "learning_rate_init",
    "batch_size",
    "max_iter",
    "early_stopping",
    "n_iter_no_change",
    "tol",
]
MLP_PARAM_COLUMNS_FOR_DISPLAY = MLP_PARAM_COLUMNS


def prepare_train_eval_feature_frames(
    features: list[str],
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if "google_trends_above_ticker_train_median" in features:
        return FeatureFrameBuilder.add_google_trends_train_median_feature(train_input_df, eval_input_df)
    return train_input_df, eval_input_df


def normalize_hidden_layer_sizes(value) -> tuple[int, ...]:
    if isinstance(value, tuple):
        return tuple(int(item) for item in value)
    if isinstance(value, list):
        return tuple(int(item) for item in value)
    if isinstance(value, np.ndarray):
        return tuple(int(item) for item in value.tolist())
    return (int(value),)


def params_from_result_row(row: dict | pd.Series) -> dict:
    return {
        "param_set": row["param_set"],
        "hidden_layer_sizes": normalize_hidden_layer_sizes(row["hidden_layer_sizes"]),
        "activation": row["activation"],
        "solver": row["solver"],
        "alpha": float(row["alpha"]),
        "learning_rate_init": float(row["learning_rate_init"]),
        "batch_size": int(row["batch_size"]),
        "max_iter": int(row["max_iter"]),
        "early_stopping": bool(row["early_stopping"]),
        "n_iter_no_change": int(row["n_iter_no_change"]),
        "tol": float(row["tol"]),
    }


def best_threshold_for_balanced_accuracy(y_true: pd.Series, scores: np.ndarray) -> tuple[float, float]:
    return ClassificationMetrics.best_threshold_for_balanced_accuracy(
        y_true,
        scores,
        min_quantile=CONFIG["threshold_min_quantile"],
        max_quantile=CONFIG["threshold_max_quantile"],
        grid_size=CONFIG["threshold_grid_size"],
        default_threshold=0.5,
    )


def metrics_from_scores(y_true: pd.Series, scores: np.ndarray, threshold: float) -> dict:
    return ClassificationMetrics.metrics_from_scores(y_true, scores, threshold)


def add_param_columns(row: dict, params: dict, param_columns: list[str]) -> None:
    for column in param_columns:
        row[column] = params.get(column)


def evaluate_mlp_params(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    train_input_df: pd.DataFrame,
    eval_input_df: pd.DataFrame,
    split_name: str,
    decision_threshold: float | None = None,
    tune_threshold: bool = False,
    return_predictions: bool = False,
) -> dict | tuple[dict, pd.DataFrame]:
    train_features_df, eval_features_df = prepare_train_eval_feature_frames(
        features,
        train_input_df,
        eval_input_df,
    )

    pipeline = build_mlp_pipeline_from_params(params)
    pipeline.fit(train_features_df[features], train_features_df["target"])
    scores = pipeline.predict_proba(eval_features_df[features])[:, 1]
    fitted_model = pipeline.named_steps["model"]

    if tune_threshold:
        decision_threshold, _ = best_threshold_for_balanced_accuracy(
            eval_features_df["target"],
            scores,
        )
    elif decision_threshold is None:
        decision_threshold = 0.5

    metric_result = metrics_from_scores(eval_features_df["target"], scores, float(decision_threshold))
    preds = metric_result.pop("preds")
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    row = {
        "split": split_name,
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "param_set": params["param_set"],
        "n_features": len(features),
        "decision_threshold": float(decision_threshold),
        **metric_result,
    }
    add_param_columns(row, params, MLP_PARAM_COLUMNS_FOR_DISPLAY)

    if not return_predictions:
        return row

    predictions_df = eval_features_df[["date", "ticker", "target"]].copy()
    predictions_df["score"] = scores
    predictions_df["prediction"] = preds
    predictions_df["decision_threshold"] = float(decision_threshold)
    return row, predictions_df


def evaluate_mlp_config_walk_forward(
    *,
    feature_set_name: str,
    features: list[str],
    params: dict,
    modeled_input_df: pd.DataFrame,
    fold_specs: list[dict],
) -> tuple[dict, list[dict]]:
    fold_rows = []
    for spec in fold_specs:
        fold_row = evaluate_mlp_params(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            train_input_df=subset_by_dates(modeled_input_df, spec["train_dates"]),
            eval_input_df=subset_by_dates(modeled_input_df, spec["validation_dates"]),
            split_name="walk_forward_validation",
            tune_threshold=CONFIG["tune_decision_threshold"],
        )
        fold_rows.append(
            {
                **fold_row,
                "fold": spec["fold"],
                "fold_train_n_dates": spec["train_n_dates"],
                "fold_validation_n_dates": spec["validation_n_dates"],
                "fold_train_date_min": spec["train_date_min"],
                "fold_train_date_max": spec["train_date_max"],
                "fold_validation_date_min": spec["validation_date_min"],
                "fold_validation_date_max": spec["validation_date_max"],
            }
        )

    fold_results_df = pd.DataFrame(fold_rows)
    metadata = FEATURE_SET_METADATA.get(feature_set_name, {})
    primary_metric = CONFIG["primary_validation_metric"]
    metric_mean = float(fold_results_df[primary_metric].mean())
    summary_row = {
        "split": "walk_forward_validation",
        "feature_set": feature_set_name,
        "feature_family": metadata.get("feature_family"),
        "param_set": params["param_set"],
        "n_features": len(features),
        "balanced_accuracy": metric_mean,
        "accuracy": float(fold_results_df["accuracy"].mean()),
        "f1_score": float(fold_results_df["f1_score"].mean()),
    }
    add_param_columns(summary_row, params, MLP_PARAM_COLUMNS_FOR_DISPLAY)
    return summary_row, fold_rows



In [9]:
selection_metric = CONFIG["selection_metric"]
walk_forward_grid_rows = []
walk_forward_fold_rows = []

for feature_set_name in FEATURE_SETS_TO_TEST:
    features = FEATURE_SETS[feature_set_name]
    for params in MLP_PARAM_GRID:
        summary_row, fold_rows = evaluate_mlp_config_walk_forward(
            feature_set_name=feature_set_name,
            features=features,
            params=params,
            modeled_input_df=modeled_df,
            fold_specs=walk_forward_fold_specs,
        )
        walk_forward_grid_rows.append(summary_row)
        walk_forward_fold_rows.extend(fold_rows)

walk_forward_grid_results_df = pd.DataFrame(walk_forward_grid_rows)
walk_forward_fold_results_df = pd.DataFrame(walk_forward_fold_rows)
if selection_metric not in walk_forward_grid_results_df.columns:
    raise KeyError(f"Selection metric is not available: {selection_metric}")

validation_grid_results_df = walk_forward_grid_results_df.sort_values(
    [selection_metric, "balanced_accuracy", "f1_score", "accuracy", "feature_set", "param_set"],
    ascending=[False, False, False, False, True, True],
).reset_index(drop=True)


In [10]:
validation_best_by_feature_set_df = ModelReportBuilder.select_best_validation_by_feature_set(
    validation_grid_results_df,
    selection_metric=CONFIG["selection_metric"],
)

validation_best_by_feature_set_report_df = ModelReportBuilder.build_validation_best_by_feature_set_report(
    validation_best_by_feature_set_df,
    param_columns=MLP_PARAM_COLUMNS_FOR_DISPLAY,
)

validation_best_by_feature_set_report_df


,feature_family,feature_set,n_features,param_set,hidden_layer_sizes,activation,solver,alpha,learning_rate_init,batch_size,max_iter,early_stopping,n_iter_no_change,tol,validation_accuracy,validation_balanced_accuracy,validation_f1_score
0,price + volume + GDELT sentiment + Reddit + Go...,Model P - price + volume + GDELT sentiment + R...,8,mlp_8_alpha1e-3_lr1e-3_b128,"(8,)",relu,adam,0.001,0.0010,128,500,False,25,0.0001,0.541553,0.546007,0.525027
1,price + volume + GDELT attention,Model J - price + volume + GDELT attention | G...,7,mlp_16_8_alpha1e-3_lr1e-3_b128,"(16, 8)",relu,adam,0.001,0.0010,128,500,False,25,0.0001,0.536379,0.544545,0.563538
2,price + volume + GDELT + Reddit,Model D - price + volume + GDELT + Reddit | la...,9,mlp_32_16_alpha1e-3_lr5e-4_b128,"(32, 16)",relu,adam,0.001,0.0005,128,500,False,25,0.0001,0.547034,0.541671,0.595274
3,price + volume + GDELT sentiment + Reddit + Go...,Model P - price + volume + GDELT sentiment + R...,8,mlp_16_8_alpha1e-3_lr1e-3_b128,"(16, 8)",relu,adam,0.001,0.0010,128,500,False,25,0.0001,0.549151,0.541039,0.469967
4,price + volume + Reddit,Model E - price + volume + Reddit | Reddit sen...,7,mlp_32_16_alpha1e-3_lr5e-4_b128,"(32, 16)",relu,adam,0.001,0.0005,128,500,False,25,0.0001,0.538758,0.540169,0.441620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,price + volume + Reddit sentiment lags + GDELT...,Model W - price + volume + Reddit sentiment la...,10,mlp_16_alpha1e-3_lr1e-3_b128,"(16,)",relu,adam,0.001,0.0010,128,500,False,25,0.0001,0.536154,0.524009,0.521318
76,price + volume + Reddit,Model E - price + volume + Reddit | Reddit sen...,8,mlp_32_16_alpha1e-3_lr5e-4_b128,"(32, 16)",relu,adam,0.001,0.0005,128,500,False,25,0.0001,0.515672,0.523633,0.358989
77,price + volume + Reddit sentiment lags,Model R - price + volume + Reddit sentiment la...,8,mlp_32_16_alpha1e-3_lr5e-4_b128,"(32, 16)",relu,adam,0.001,0.0005,128,500,False,25,0.0001,0.515672,0.523633,0.358989
78,price + volume + all alternative data,Model F - price + volume + all alternative dat...,10,mlp_8_alpha1e-3_lr1e-3_b128,"(8,)",relu,adam,0.001,0.0010,128,500,False,25,0.0001,0.523689,0.522593,0.569637


In [11]:
best_validation_params_df = validation_best_by_feature_set_df.copy()


In [12]:
threshold_calibration_rows = []
test_rows = []

for row in best_validation_params_df.to_dict(orient="records"):
    params = params_from_result_row(row)
    feature_set_name = row["feature_set"]
    calibration_row = evaluate_mlp_params(
        feature_set_name=feature_set_name,
        features=FEATURE_SETS[feature_set_name],
        params=params,
        train_input_df=train_df,
        eval_input_df=validation_df,
        split_name="validation_threshold_calibration",
        tune_threshold=True,
    )
    threshold_calibration_rows.append(calibration_row)
    test_rows.append(
        evaluate_mlp_params(
            feature_set_name=feature_set_name,
            features=FEATURE_SETS[feature_set_name],
            params=params,
            train_input_df=train_validation_df,
            eval_input_df=test_df,
            split_name="test_refit_train_validation_selected_threshold",
            decision_threshold=calibration_row["decision_threshold"],
            tune_threshold=False,
        )
    )

threshold_calibration_results_df = pd.DataFrame(threshold_calibration_rows)
test_best_validation_params_df = pd.DataFrame(test_rows).sort_values(
    ["balanced_accuracy", "f1_score", "accuracy", "feature_set"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

(
    simple_hyperparameter_summary_df,
    baseline_walk_forward_row,
    baseline_calibration_row,
    baseline_test_row,
) = ModelReportBuilder.build_simple_hyperparameter_summary(
    best_validation_params_df=best_validation_params_df,
    threshold_calibration_results_df=threshold_calibration_results_df,
    test_best_validation_params_df=test_best_validation_params_df,
    baseline_feature_set=BASELINE_FEATURE_SET,
)


In [13]:
validation_selected_family_test_report_df = ModelReportBuilder.build_validation_selected_family_test_report(
    simple_hyperparameter_summary_df,
    param_columns=MLP_PARAM_COLUMNS_FOR_DISPLAY,
)

final_test_verification_path = ModelReportBuilder.save_final_test_verification(
    validation_selected_family_test_report_df,
    model_name="neutral_net",
    output_dir=PROJECT_ROOT / "notebooks" / "outputs",
)
print(f"Saved final test verification to: {final_test_verification_path}")

validation_selected_family_test_report_df


Saved final test verification to: C:\Users\user\OneDrive\Documents\magisterka\praca magisterska\code\notebooks\outputs\neutral_net_final_test_verification.csv


,feature_family,feature_set,n_features,param_set,hidden_layer_sizes,activation,solver,alpha,learning_rate_init,batch_size,max_iter,early_stopping,n_iter_no_change,tol,validation_accuracy,validation_balanced_accuracy,validation_f1_score,test_accuracy,test_balanced_accuracy,test_f1_score,price_volume_baseline_feature_set,price_volume_baseline_test_balanced_accuracy,test_balanced_accuracy_change_vs_price_volume
0,price + volume + GDELT sentiment + Reddit + Go...,Model P - price + volume + GDELT sentiment + R...,8,mlp_8_alpha1e-3_lr1e-3_b128,"(8,)",relu,adam,0.001,0.0010,128,500,False,25,0.0001,0.541553,0.546007,0.525027,0.540297,0.528992,0.629645,Model B - price + volume | volume zscore 10d,0.505114,0.023878
1,price + volume + GDELT + Reddit attention,Model M - price + volume + GDELT + Reddit atte...,7,mlp_16_8_alpha1e-3_lr1e-3_b128,"(16, 8)",relu,adam,0.001,0.0010,128,500,False,25,0.0001,0.545220,0.532957,0.419534,0.538176,0.520212,0.662010,Model B - price + volume | volume zscore 10d,0.505114,0.015098
2,price + volume + GDELT sentiment lags + Google...,Model X - price + volume + GDELT sentiment lag...,10,mlp_8_alpha1e-3_lr1e-3_b128,"(8,)",relu,adam,0.001,0.0010,128,500,False,25,0.0001,0.537169,0.534501,0.427408,0.513786,0.519943,0.466550,Model B - price + volume | volume zscore 10d,0.505114,0.014828
3,price + volume + Google attention lags,Model U - price + volume + Google attention la...,8,mlp_32_16_alpha1e-3_lr5e-4_b128,"(32, 16)",relu,adam,0.001,0.0005,128,500,False,25,0.0001,0.536092,0.530121,0.429133,0.504772,0.519897,0.332857,Model B - price + volume | volume zscore 10d,0.505114,0.014783
4,price + volume + Reddit + Google,Model I - price + volume + Reddit + Google | z...,8,mlp_16_8_alpha1e-3_lr1e-3_b128,"(16, 8)",relu,adam,0.001,0.0010,128,500,False,25,0.0001,0.529564,0.539268,0.420382,0.538706,0.515190,0.686373,Model B - price + volume | volume zscore 10d,0.505114,0.010076
5,price + volume + GDELT attention lags,Model S - price + volume + GDELT attention lag...,8,mlp_16_8_alpha1e-3_lr1e-3_b128,"(16, 8)",relu,adam,0.001,0.0010,128,500,False,25,0.0001,0.524005,0.528198,0.527463,0.507423,0.514746,0.446035,Model B - price + volume | volume zscore 10d,0.505114,0.009632
6,price + volume + GDELT sentiment + Reddit atte...,Model O - price + volume + GDELT sentiment + R...,7,mlp_8_alpha1e-3_lr1e-3_b128,"(8,)",relu,adam,0.001,0.0010,128,500,False,25,0.0001,0.553243,0.528878,0.467203,0.533934,0.512237,0.675526,Model B - price + volume | volume zscore 10d,0.505114,0.007123
7,price + volume + GDELT sentiment lags,Model Q - price + volume + GDELT sentiment lag...,8,mlp_32_16_alpha1e-2_lr5e-4_b256,"(32, 16)",relu,adam,0.010,0.0005,256,500,False,25,0.0001,0.507333,0.526452,0.494106,0.532344,0.510106,0.676686,Model B - price + volume | volume zscore 10d,0.505114,0.004992
8,price + volume + GDELT sentiment lags + Reddit...,Model V - price + volume + GDELT sentiment lag...,10,mlp_16_alpha1e-2_lr1e-3_b128,"(16,)",relu,adam,0.010,0.0010,128,500,False,25,0.0001,0.510491,0.530622,0.492361,0.484624,0.509221,0.084746,Model B - price + volume | volume zscore 10d,0.505114,0.004107
9,price + volume + GDELT attention,Model J - price + volume + GDELT attention | G...,7,mlp_16_8_alpha1e-3_lr1e-3_b128,"(16, 8)",relu,adam,0.001,0.0010,128,500,False,25,0.0001,0.536379,0.544545,0.563538,0.507953,0.508765,0.513627,Model B - price + volume | volume zscore 10d,0.505114,0.003651
